# Data Merger
Parses, cleans, and left-joins three raw data sources — **Price**, **Macro**, and **Sentiment** — into a single ticker-level `_merged.csv` for every ticker in the universe.

| Step | Source | Output |
|---|---|---|
| Cell 2 | `data/raw_macro/` CSVs | `macro_df` (universal daily table) |
| Cell 3 | `data/raw_sentiment/` `.jsonl.gz` | per-ticker `sentiment_df` |
| Cell 4 | `data/raw_price/` CSVs + above | `data/merged_raw/[TICKER]_merged.csv` |

> **No ML features are engineered here.** This notebook produces the aligned base dataset only.

## Cell 1 — Setup & Imports

In [14]:
import gzip
import json
import os
from glob import glob

import numpy as np
import pandas as pd

from config import MACRO_YFINANCE_MAP, TICKERS

# ---------------------------------------------------------------------------
# Path constants
# ---------------------------------------------------------------------------
RAW_PRICE_DIR: str = "data/raw_price"
RAW_MACRO_DIR: str = "data/raw_macro"
RAW_SENTIMENT_DIR: str = "data/raw_sentiment"
MERGED_DIR: str = "data/merged_raw"
SPY_PRICE_PATH: str = os.path.join(RAW_PRICE_DIR, "SPY_daily.csv")

# ---------------------------------------------------------------------------
# Sentiment filtering & aggregation constants
# ---------------------------------------------------------------------------
RELEVANCE_THRESHOLD: float = 0.5
SENTIMENT_COLS: list[str] = [
    "article_count", "avg_sentiment", "weighted_avg_sentiment",
    "max_sentiment", "min_sentiment",
]

os.makedirs(MERGED_DIR, exist_ok=True)

print(f"Ticker universe : {len(TICKERS)} tickers")
print(f"Output dir      : {MERGED_DIR}")

Ticker universe : 50 tickers
Output dir      : data/merged_raw


## Cell 2 — Macro Aggregator
Reads `ES=F`, `NQ=F`, `^TNX`, and `SPY` from `data/raw_macro/`, keeps only the close column for each, and outer-joins them on the Date index into a single `macro_df`. Forward-fills gaps caused by holidays/weekends.

In [15]:
# yfinance CSVs have 3 header rows:
#   Row 0 → field names  (Price, Close, High, Low, Open, Volume)
#   Row 1 → ticker names (Ticker, ES=F, ES=F, …)   ← skip
#   Row 2 → "Date" label (Date, , , , , )           ← skip
# Skipping rows 1 & 2 leaves row 0 as the column header and dates as the index.

# MACRO_YFINANCE_MAP and SPY_PRICE_PATH are defined in Cell 1.


def _load_yfinance_series(file_stem: str, col_name: str) -> pd.Series:
    """
    Load a yfinance macro CSV (3-row header format) and return a named
    daily close Series with a timezone-naive DatetimeIndex.

    Parameters
    ----------
    file_stem : str
        Filename stem without '_daily.csv' (e.g. 'ES_F').
    col_name : str
        Name to assign to the resulting Series.

    Returns
    -------
    pd.Series
        Daily close values with DatetimeIndex named 'Date'.
    """
    path: str = os.path.join(RAW_MACRO_DIR, f"{file_stem}_daily.csv")
    if not os.path.exists(path):
        print(f"  [WARN] Macro file not found, skipping: {path}")
        return pd.Series(dtype=float, name=col_name)

    # skiprows=[1, 2] drops the 'Ticker' and 'Date' label rows so that
    # row 0 (field names) becomes the column header cleanly.
    df: pd.DataFrame = pd.read_csv(
        path, index_col=0, skiprows=[1, 2], parse_dates=True
    )
    df.index.name = "Date"
    df.index = pd.to_datetime(df.index).tz_localize(None)

    close_candidates: list[str] = ["Close", "close", "Adj Close", "adjusted close"]
    close_col: str | None = next(
        (c for c in close_candidates if c in df.columns), None
    )
    if close_col is None:
        raise KeyError(
            f"Could not locate a close column in {path}. "
            f"Available columns: {df.columns.tolist()}"
        )

    return df[close_col].rename(col_name)


def _load_av_series(path: str, col_name: str) -> pd.Series:
    """
    Load an Alpha Vantage daily-adjusted CSV and return a named adjusted-close
    Series with a timezone-naive DatetimeIndex.

    Alpha Vantage CSVs use 'timestamp' as the date column and
    'adjusted close' as the relevant price column.

    Parameters
    ----------
    path : str
        Full file path to the CSV.
    col_name : str
        Name to assign to the resulting Series.

    Returns
    -------
    pd.Series
        Daily adjusted-close values with DatetimeIndex named 'Date'.
    """
    if not os.path.exists(path):
        print(f"  [WARN] AV file not found, skipping: {path}")
        return pd.Series(dtype=float, name=col_name)

    df: pd.DataFrame = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index.name = "Date"
    df.index = pd.to_datetime(df.index).tz_localize(None)

    close_candidates: list[str] = ["adjusted close", "Adj Close", "Close", "close"]
    close_col: str | None = next(
        (c for c in close_candidates if c in df.columns), None
    )
    if close_col is None:
        raise KeyError(
            f"Could not locate a close column in {path}. "
            f"Available columns: {df.columns.tolist()}"
        )

    return df[close_col].rename(col_name)


# ---------------------------------------------------------------------------
# Build the universal macro DataFrame
# ---------------------------------------------------------------------------
macro_series_list: list[pd.Series] = [
    _load_yfinance_series(stem, col)
    for stem, col in MACRO_YFINANCE_MAP.items()
]
macro_series_list.append(_load_av_series(SPY_PRICE_PATH, "SPY_AdjClose"))

macro_df: pd.DataFrame = (
    pd.concat(macro_series_list, axis=1, join="outer")
    .sort_index()
    .ffill()
)
macro_df.index.name = "Date"

print(f"macro_df shape  : {macro_df.shape}")
print(f"Date range      : {macro_df.index.min().date()} → {macro_df.index.max().date()}")
macro_df.head(3)

macro_df shape  : (6673, 4)
Date range      : 1999-11-01 → 2026-05-08


,ESF_Close,NQF_Close,TNX_Close,SPY_AdjClose
Date,,,,
1999-11-01,NaN,NaN,NaN,135.562500
1999-11-02,NaN,NaN,NaN,134.593704
1999-11-03,NaN,NaN,NaN,135.500000


## Cell 3 — Sentiment Parser & Aggregator
Defines `process_sentiment_for_ticker()`, which:
1. Globs all `.jsonl.gz` files for a given ticker across every year/month subdirectory.
2. Parses each line as a raw Alpha Vantage `NEWS_SENTIMENT` JSON response.
3. Extracts `time_published`, `ticker_sentiment_score`, and `relevance_score` for the target ticker.
4. Filters articles with `relevance_score < 0.8`.
5. Aggregates to daily `article_count`, `avg_sentiment`, `max_sentiment`, `min_sentiment`.

In [16]:
def process_sentiment_for_ticker(ticker: str) -> pd.DataFrame:
    """
    Parse all sentiment `.jsonl.gz` archives for a single ticker and return
    a daily aggregated DataFrame.

    File format
    -----------
    Each line in the `.jsonl.gz` is a **single article** JSON object with the
    following relevant fields:
      - time_published        : str  e.g. '20240115T143000'
      - ticker_sentiment      : list of dicts, one per mentioned ticker:
            { 'ticker': str, 'relevance_score': str, 'ticker_sentiment_score': str }

    Note: the key is `ticker_sentiment` (singular), NOT `ticker_sentiments`.

    Parameters
    ----------
    ticker : str
        Equity ticker symbol (e.g. 'NVDA').

    Returns
    -------
    pd.DataFrame
        Columns: article_count, avg_sentiment, weighted_avg_sentiment,
                 max_sentiment, min_sentiment.
        Index  : DatetimeIndex named 'Date' (one row per calendar day with data).
        Returns an empty DataFrame with the above columns if no files are found.

        weighted_avg_sentiment is computed as:
            sum(sentiment_score * relevance_score) / sum(relevance_score)
        giving higher-relevance articles proportionally more influence.
    """
    pattern: str = os.path.join(RAW_SENTIMENT_DIR, "**", f"{ticker}_*.jsonl.gz")
    files: list[str] = sorted(glob(pattern, recursive=True))

    if not files:
        return pd.DataFrame(columns=SENTIMENT_COLS)

    records: list[dict] = []

    for filepath in files:
        try:
            with gzip.open(filepath, "rt", encoding="utf-8") as fh:
                for raw_line in fh:
                    raw_line = raw_line.strip()
                    if not raw_line:
                        continue
                    try:
                        # Each line is a single article object (not a feed wrapper)
                        article: dict = json.loads(raw_line)
                    except json.JSONDecodeError:
                        continue

                    time_published: str = article.get("time_published", "")
                    if len(time_published) < 8:
                        continue
                    date_str: str = (
                        f"{time_published[:4]}-"
                        f"{time_published[4:6]}-"
                        f"{time_published[6:8]}"
                    )

                    # Key is 'ticker_sentiment' (singular)
                    for ts in article.get("ticker_sentiment", []):
                        if ts.get("ticker") != ticker:
                            continue
                        try:
                            relevance: float = float(ts.get("relevance_score", 0))
                            sentiment: float = float(
                                ts.get("ticker_sentiment_score", 0)
                            )
                        except (ValueError, TypeError):
                            continue

                        if relevance < RELEVANCE_THRESHOLD:
                            continue

                        records.append(
                            {
                                "Date": date_str,
                                "sentiment_score": sentiment,
                                "relevance_score": relevance,
                            }
                        )
        except (OSError, gzip.BadGzipFile) as exc:
            print(f"  [WARN] Could not read {filepath}: {exc}")

    if not records:
        return pd.DataFrame(columns=SENTIMENT_COLS)

    raw: pd.DataFrame = pd.DataFrame(records)
    raw["Date"] = pd.to_datetime(raw["Date"])

    # ------------------------------------------------------------------
    # Standard aggregations: count, mean, max, min
    # ------------------------------------------------------------------
    agg: pd.DataFrame = raw.groupby("Date")["sentiment_score"].agg(
        article_count="count",
        avg_sentiment="mean",
        max_sentiment="max",
        min_sentiment="min",
    )

    # ------------------------------------------------------------------
    # Relevance-weighted average sentiment
    #   weighted_avg = sum(sentiment * relevance) / sum(relevance)
    # This down-weights tangentially-mentioned tickers (low relevance) and
    # amplifies articles where the ticker is the primary subject.
    # ------------------------------------------------------------------
    raw["weighted_contribution"] = raw["sentiment_score"] * raw["relevance_score"]

    weighted: pd.Series = (
        raw.groupby("Date")["weighted_contribution"].sum()
        / raw.groupby("Date")["relevance_score"].sum()
    ).rename("weighted_avg_sentiment")

    agg = agg.join(weighted)
    agg.index.name = "Date"

    # Reorder columns to match SENTIMENT_COLS
    return agg[SENTIMENT_COLS]


# Smoke-test with the first ticker
_sample: pd.DataFrame = process_sentiment_for_ticker(TICKERS[0])
print(f"Sentiment smoke-test ({TICKERS[0]}): {len(_sample)} days with data")
_sample.head(3)

Sentiment smoke-test (NVDA): 654 days with data


,article_count,avg_sentiment,weighted_avg_sentiment,max_sentiment,min_sentiment
Date,,,,,
2024-05-08,4,0.187131,0.194300,0.435717,-0.204257
2024-05-09,3,0.353158,0.360247,0.411155,0.305288
2024-05-10,4,0.329563,0.318642,0.500144,0.109756


## Cell 4 — Grand Merger Loop (Inner Join with Zero-Fill)
For each ticker:
1. Loads price CSV → timezone-naive DatetimeIndex.
2. Calls `process_sentiment_for_ticker()` → daily sentiment (timezone-naive index).
3. **Weekend roll-forward**: any weekend sentiment rows are merged forward to the next Monday so no news is lost before trading-day alignment.
4. **Zero-fill alignment**: reindexes `sentiment_df` to the exact slice of price trading days that falls within the sentiment date range, then `fillna(0)` fills all news-silent days.
5. **Inner-joins** price ← sentiment on the now-perfectly-aligned index.
6. Left-joins the result ← `macro_df`.
7. Saves to `data/merged_raw/[TICKER]_merged.csv`.

In [17]:
skipped: list[str] = []
total: int = len(TICKERS)

for idx, ticker in enumerate(TICKERS, start=1):
    print(f"[{idx:>3}/{total}] {ticker} ", end="")

    # ------------------------------------------------------------------
    # 1. Load price data — timezone-naive DatetimeIndex
    # ------------------------------------------------------------------
    price_path: str = os.path.join(RAW_PRICE_DIR, f"{ticker}_daily.csv")
    if not os.path.exists(price_path):
        print("→ [SKIP] price file not found")
        skipped.append(ticker)
        continue

    # Guard: detect files that contain an Alpha Vantage error response
    # instead of real OHLCV data (happens when the fetcher saved a bad API reply).
    with open(price_path, "r", encoding="utf-8") as _fh:
        _first_line: str = _fh.readline()
    if "Error Message" in _first_line or "Invalid API" in _first_line or "Thank you" in _first_line:
        print(f"→ [SKIP] price file contains API error response — re-run ohlcv_fetcher for this ticker")
        skipped.append(ticker)
        continue

    try:
        price_df: pd.DataFrame = pd.read_csv(
            price_path, index_col="timestamp", parse_dates=True
        )
    except (ValueError, KeyError):
        price_df = pd.read_csv(price_path, index_col=0, parse_dates=True)

    price_df.index.name = "Date"
    price_df.index = pd.to_datetime(price_df.index).tz_localize(None)
    price_df.sort_index(inplace=True)

    # ------------------------------------------------------------------
    # 2. Build sentiment DataFrame — ensure timezone-naive index
    # ------------------------------------------------------------------
    sentiment_df: pd.DataFrame = process_sentiment_for_ticker(ticker)

    if sentiment_df.empty:
        print("→ [SKIP] no sentiment data found")
        skipped.append(ticker)
        continue

    sentiment_df.index = pd.to_datetime(sentiment_df.index).tz_localize(None)
    sentiment_df.sort_index(inplace=True)

    # ------------------------------------------------------------------
    # 3. Weekend roll-forward
    #    Aggregate any Saturday/Sunday rows forward to the next Monday so
    #    weekend news contributes to the following trading day rather than
    #    being silently dropped by the inner join.
    # ------------------------------------------------------------------
    sentiment_df = (
        sentiment_df
        .resample("B")           # business-day grid
        .agg(
            article_count=("article_count", "sum"),
            avg_sentiment=("avg_sentiment",  "mean"),
            max_sentiment=("max_sentiment",  "max"),
            min_sentiment=("min_sentiment",  "min"),
        )
        .replace(0, np.nan)      # turn 0-fill artefacts back to NaN so ffill works
        .ffill(limit=3)          # roll weekend articles forward (max 3 calendar days)
        .dropna(how="all")       # drop days that were never populated
    )

    # ------------------------------------------------------------------
    # 4. Zero-fill alignment
    #    Reindex sentiment to every trading day that sits inside the
    #    sentiment data's own date range, then zero-fill the gaps.
    # ------------------------------------------------------------------
    sent_min: pd.Timestamp = sentiment_df.index.min()
    sent_max: pd.Timestamp = sentiment_df.index.max()

    windowed_price_index: pd.DatetimeIndex = price_df.index[
        (price_df.index >= sent_min) & (price_df.index <= sent_max)
    ]

    sentiment_df = sentiment_df.reindex(windowed_price_index).fillna(0)
    sentiment_df["article_count"] = sentiment_df["article_count"].astype(int)

    # ------------------------------------------------------------------
    # 5. Inner join: price ← sentiment
    #    Both indices are now identical trading days — no rows are lost.
    # ------------------------------------------------------------------
    merged: pd.DataFrame = price_df.merge(
        sentiment_df,
        how="inner",
        left_index=True,
        right_index=True,
    )

    # ------------------------------------------------------------------
    # 6. Left join: merged ← macro
    # ------------------------------------------------------------------
    merged = merged.join(macro_df, how="left")

    # ------------------------------------------------------------------
    # 7. Persist
    # ------------------------------------------------------------------
    out_path: str = os.path.join(MERGED_DIR, f"{ticker}_merged.csv")
    merged.to_csv(out_path)
    print(f"→ {merged.shape[0]} rows × {merged.shape[1]} cols  →  {out_path}")

# Summary
saved: int = total - len(skipped)
print(f"\nDone. {saved}/{total} tickers merged.")
if skipped:
    print(f"Skipped: {skipped}")

[  1/50] NVDA → 501 rows × 16 cols  →  data/merged_raw/NVDA_merged.csv
[  2/50] AMD → 501 rows × 16 cols  →  data/merged_raw/AMD_merged.csv
[  3/50] TSM → 499 rows × 16 cols  →  data/merged_raw/TSM_merged.csv
[  4/50] AVGO → 501 rows × 16 cols  →  data/merged_raw/AVGO_merged.csv
[  5/50] MU → 501 rows × 16 cols  →  data/merged_raw/MU_merged.csv
[  6/50] INTC → 501 rows × 16 cols  →  data/merged_raw/INTC_merged.csv
[  7/50] ARM → 500 rows × 16 cols  →  data/merged_raw/ARM_merged.csv
[  8/50] QCOM → 501 rows × 16 cols  →  data/merged_raw/QCOM_merged.csv
[  9/50] ASML → 501 rows × 16 cols  →  data/merged_raw/ASML_merged.csv
[ 10/50] SMCI → 500 rows × 16 cols  →  data/merged_raw/SMCI_merged.csv
[ 11/50] MRVL → 486 rows × 16 cols  →  data/merged_raw/MRVL_merged.csv
[ 12/50] TXN → 501 rows × 16 cols  →  data/merged_raw/TXN_merged.csv
[ 13/50] KLAC → 496 rows × 16 cols  →  data/merged_raw/KLAC_merged.csv
[ 14/50] AMAT → 501 rows × 16 cols  →  data/merged_raw/AMAT_merged.csv
[ 15/50] LRCX → 50